## 1. Environment Setup

Install required packages for ONNX export and model optimization.

In [ ]:
# Install required packages
import subprocess
import sys

packages = [
    'ultralytics>=8.3.0',  # YOLO11 with ONNX export
    'onnx>=1.14.0',        # ONNX core library
    'onnxruntime',         # ONNX runtime for validation
    'onnxsim',             # ONNX simplification
]

print("📦 Installing required packages...")
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print("✅ Installation complete!")

## 2. Import Libraries

In [ ]:
from ultralytics import YOLO
import torch
import os
from pathlib import Path
import shutil
import onnx
import onnxsim
import pandas as pd

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"ONNX version: {onnx.__version__}")
print(f"Ultralytics version: {YOLO.__module__}")

## 3. Configuration

Set paths and parameters for your trained model and export settings.

In [ ]:
# Model Configuration
MODEL_PATH = "runs/detect/yolo11n_leaf_esp32/weights/best.pt"
OUTPUT_DIR = "esp32_onnx_models"
DATA_YAML = "Dataset.v1.yolov11/data.yaml"

# ESP32-S3 optimized input sizes
# Format: (size, description, expected_inference_time)
ESP32_CONFIGS = [
    (128, "Maximum speed", "1.5-2.5s"),
    (160, "Recommended balance", "2-4s"),
    (192, "Better accuracy", "4-6s"),
]

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"✅ Configuration:")
print(f"   Model: {MODEL_PATH}")
print(f"   Output: {OUTPUT_DIR}")
print(f"   Dataset: {DATA_YAML}")
print(f"   Export sizes: {[cfg[0] for cfg in ESP32_CONFIGS]}")

## 4. Load and Verify Model

In [ ]:
print("🔍 Loading and verifying model...")

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"❌ Model not found: {MODEL_PATH}")

# Load model
model = YOLO(MODEL_PATH)

# Get model information
model_size_mb = os.path.getsize(MODEL_PATH) / (1024 * 1024)
num_params = sum(p.numel() for p in model.model.parameters()) / 1e6

print(f"\n✅ Model loaded successfully!")
print(f"   Architecture: YOLO11n")
print(f"   Size: {model_size_mb:.2f} MB")
print(f"   Parameters: {num_params:.2f}M")
print(f"   Classes: {model.names}")
print(f"   Number of classes: {len(model.names)}")

## 5. Export to ONNX

Export models in multiple sizes optimized for ESP32-S3 deployment.

**Note:** We use `simplify=False` during export to avoid kernel crashes, then manually simplify using onnxsim (safer approach).

In [ ]:
print("="*80)
print("ONNX EXPORT FOR ESP32-S3")
print("="*80)

exported_models = {}

for imgsz, description, inference_time in ESP32_CONFIGS:
    print(f"\n📦 Exporting ONNX @ {imgsz}×{imgsz}")
    print(f"   Profile: {description}")
    print(f"   Expected inference: {inference_time}")
    
    try:
        # Step 1: Export to ONNX (without simplification to avoid crashes)
        print(f"   ⏳ Exporting base ONNX...")
        onnx_path = model.export(
            format="onnx",
            imgsz=imgsz,
            simplify=False,    # Avoid onnxslim - prevents kernel crash
            opset=12,          # Compatible with ESP-PPQ and converters
            dynamic=False,     # Static shapes for embedded deployment
        )
        
        if not onnx_path or not os.path.exists(str(onnx_path)):
            print(f"   ❌ Export failed - file not created")
            continue
        
        # Step 2: Copy to output directory
        onnx_name = f"yolo11n_{imgsz}.onnx"
        onnx_output = os.path.join(OUTPUT_DIR, onnx_name)
        shutil.copy(str(onnx_path), onnx_output)
        
        base_size = os.path.getsize(onnx_output) / (1024 * 1024)
        print(f"   ✅ Base ONNX: {base_size:.2f} MB")
        
        # Step 3: Manually simplify using onnxsim (safer than built-in)
        print(f"   ⏳ Simplifying ONNX...")
        try:
            model_onnx = onnx.load(onnx_output)
            model_simplified, check = onnxsim.simplify(
                model_onnx,
                check_n=3,
                perform_optimization=True,
            )
            
            if check:
                simplified_path = onnx_output.replace('.onnx', '_simplified.onnx')
                onnx.save(model_simplified, simplified_path)
                simplified_size = os.path.getsize(simplified_path) / (1024 * 1024)
                reduction = ((base_size - simplified_size) / base_size) * 100
                
                print(f"   ✅ Simplified: {simplified_size:.2f} MB ({reduction:.1f}% reduction)")
                
                # Store both versions
                exported_models[imgsz] = {
                    'base_path': onnx_output,
                    'base_size_mb': base_size,
                    'simplified_path': simplified_path,
                    'simplified_size_mb': simplified_size,
                    'description': description,
                    'inference_time': inference_time,
                    'reduction_percent': reduction
                }
            else:
                print(f"   ⚠️  Simplification check failed, using base model")
                exported_models[imgsz] = {
                    'base_path': onnx_output,
                    'base_size_mb': base_size,
                    'description': description,
                    'inference_time': inference_time,
                }
        except Exception as simp_error:
            print(f"   ⚠️  Simplification failed: {str(simp_error)[:80]}")
            print(f"   ℹ️  Using base ONNX model")
            exported_models[imgsz] = {
                'base_path': onnx_output,
                'base_size_mb': base_size,
                'description': description,
                'inference_time': inference_time,
            }
        
        print(f"   📁 Saved to: {OUTPUT_DIR}")
        
    except Exception as e:
        print(f"   ❌ Export failed: {str(e)[:150]}")
        continue

print(f"\n{'='*80}")
print(f"✅ Export complete! {len(exported_models)} models exported")
print(f"{'='*80}")

## 6. Export Summary & Model Comparison

In [ ]:
if exported_models:
    print("\n📊 EXPORTED MODELS SUMMARY\n")
    
    # Create summary table
    summary_data = []
    for imgsz, info in exported_models.items():
        row = {
            'Input Size': f"{imgsz}×{imgsz}",
            'Description': info['description'],
            'Base Size': f"{info['base_size_mb']:.2f} MB",
            'Optimized Size': f"{info.get('simplified_size_mb', info['base_size_mb']):.2f} MB",
            'Reduction': f"{info.get('reduction_percent', 0):.1f}%",
            'ESP32 Inference': info['inference_time'],
            'Recommendation': '🏆 Best' if imgsz == 160 else ('⚡ Fast' if imgsz == 128 else '🎯 Accurate')
        }
        summary_data.append(row)
    
    df = pd.DataFrame(summary_data)
    print(df.to_string(index=False))
    
    print(f"\n📁 Output Directory: {os.path.abspath(OUTPUT_DIR)}")
    
    # List all exported files
    print(f"\n📄 Exported Files:")
    for imgsz, info in exported_models.items():
        print(f"   • {os.path.basename(info['base_path'])}")
        if 'simplified_path' in info:
            print(f"   • {os.path.basename(info['simplified_path'])} (use this for deployment)")
    
    # Recommendation
    if 160 in exported_models:
        print(f"\n🎯 RECOMMENDED FOR ESP32-S3:")
        model_path = exported_models[160].get('simplified_path', exported_models[160]['base_path'])
        model_size = exported_models[160].get('simplified_size_mb', exported_models[160]['base_size_mb'])
        print(f"   Model: {os.path.basename(model_path)}")
        print(f"   Size: {model_size:.2f} MB")
        print(f"   Inference: {exported_models[160]['inference_time']}")
        print(f"   Memory: ~{model_size * 1.5:.1f} MB (after INT8 quantization)")
        
else:
    print("\n❌ No models exported successfully")
    print("   Please check:")
    print("   1. Model file exists and is valid")
    print("   2. Ultralytics version >= 8.3.0")
    print("   3. PyTorch is properly installed")

## 7. Model Validation (Optional)

Validate the exported ONNX model with ONNX Runtime.

In [ ]:
import onnxruntime as ort
import numpy as np

def validate_onnx_model(onnx_path, imgsz=160):
    """Validate ONNX model can be loaded and run inference."""
    print(f"\n🔍 Validating: {os.path.basename(onnx_path)}")
    
    try:
        # Load ONNX model
        session = ort.InferenceSession(onnx_path)
        
        # Get input/output info
        input_name = session.get_inputs()[0].name
        input_shape = session.get_inputs()[0].shape
        output_names = [out.name for out in session.get_outputs()]
        
        print(f"   Input: {input_name} {input_shape}")
        print(f"   Outputs: {len(output_names)} tensors")
        
        # Run dummy inference
        dummy_input = np.random.randn(1, 3, imgsz, imgsz).astype(np.float32)
        outputs = session.run(None, {input_name: dummy_input})
        
        print(f"   Output shape: {outputs[0].shape}")
        print(f"   ✅ Validation successful!")
        return True
        
    except Exception as e:
        print(f"   ❌ Validation failed: {str(e)[:100]}")
        return False

# Validate the recommended model (160x160)
if 160 in exported_models:
    model_to_validate = exported_models[160].get('simplified_path', exported_models[160]['base_path'])
    validate_onnx_model(model_to_validate, imgsz=160)

## 8. ESP32-S3 Deployment Guide

### Option 1: ESP-Detection + ESP-DL (Recommended)

**Step 1: Clone ESP-Detection**
```bash
git clone https://github.com/espressif/esp-detection.git
cd esp-detection
```

**Step 2: Quantize ONNX to ESP-DL Format**
```bash
cd tools/quantization

# Install esp-ppq
pip install esp-ppq

# Quantize model to INT8
python quantize_onnx.py \
    --model_path ../../esp32_onnx_models/yolo11n_160_simplified.onnx \
    --output_path yolo11n_160_int8.espdl \
    --calibration_dataset path/to/calibration/images \
    --num_calibration 100
```

**Step 3: Deploy to ESP32-S3**
```bash
# Copy .espdl model to your ESP-IDF project
cp yolo11n_160_int8.espdl /path/to/esp32-project/model/

# Build and flash
idf.py build
idf.py flash monitor
```

### Option 2: TFLite Micro (Alternative)

**Convert ONNX to TFLite:**
```bash
# Install onnx2tf
pip install onnx2tf

# Convert to TFLite INT8
onnx2tf -i yolo11n_160_simplified.onnx -o tflite_output -oiqt

# Convert to C array for embedding
xxd -i model_int8.tflite > model_data.h
```

### Hardware Setup

**Required:**
- ESP32-S3-WROOM-1-N8R8 (8MB PSRAM) ⚠️ Mandatory
- 16MB Flash memory
- Camera module: OV2640, OV5640, or ESP32-S3-EYE

**Wiring (ESP32-S3 + OV2640):**
```
Camera Pin  →  ESP32-S3 GPIO
SIOD (SDA)  →  GPIO 40
SIOC (SCL)  →  GPIO 39
VSYNC       →  GPIO 6
HREF        →  GPIO 7
PCLK        →  GPIO 13
XCLK        →  GPIO 15
D0-D7       →  GPIO 11, 9, 8, 10, 12, 18, 17, 16
```

### Performance Expectations

| Input Size | Inference Time | Accuracy | Memory |
|------------|---------------|----------|--------|
| 128×128    | 1.5-2.5s      | Moderate | ~2.0 MB |
| 160×160    | 2-4s          | Good     | ~2.5 MB |
| 192×192    | 4-6s          | Best     | ~3.0 MB |

### Troubleshooting

**Out of Memory:**
- Ensure you have ESP32-S3 with 8MB PSRAM
- Use smaller input size (128×128)
- Reduce batch size to 1

**Slow Inference:**
- Verify INT8 quantization is applied
- Enable ESP-DL optimizations
- Use lower resolution input

**Low Accuracy:**
- Use proper calibration dataset (100+ images)
- Increase input resolution to 192×192
- Fine-tune quantization parameters

### Resources

- [ESP-Detection GitHub](https://github.com/espressif/esp-detection)
- [ESP-DL Documentation](https://docs.espressif.com/projects/esp-dl/)
- [ESP32-S3 Datasheet](https://www.espressif.com/sites/default/files/documentation/esp32-s3_datasheet_en.pdf)
- [YOLO11 Export Docs](https://docs.ultralytics.com/modes/export/)

## 9. Final Checklist

Before deployment, verify:

- [ ] ✅ Models exported successfully
- [ ] ✅ ONNX validation passed
- [ ] ✅ Simplified models generated (smaller size)
- [ ] ✅ ESP32-S3 hardware ready (8MB PSRAM)
- [ ] ✅ Camera module connected and tested
- [ ] ✅ Calibration dataset prepared (for INT8 quantization)
- [ ] ✅ ESP-IDF environment set up
- [ ] ✅ Deployment framework chosen (ESP-DL or TFLite)

---

**Next Steps:**
1. Quantize your ONNX model to INT8 using ESP-PPQ
2. Deploy to ESP32-S3 using ESP-DL framework
3. Test inference speed and accuracy
4. Optimize based on your requirements

**Support:**
- GitHub Issues: [esp-detection](https://github.com/espressif/esp-detection/issues)
- ESP32 Forum: [forum.espressif.com](https://forum.espressif.com/)
- Ultralytics Discord: [discord.gg/ultralytics](https://discord.gg/ultralytics)

---
🎉 **Export Complete!** Your YOLO11n model is ready for ESP32-S3 deployment!